# Compilation

torchode is fully compatible with [`torch.compile`](https://pytorch.org/docs/stable/torch.compiler.html). By compiling your model together with the ODE solver, PyTorch can fuse and schedule the underlying computations more efficiently and reduce Python interpreter overhead in the forward pass. How much this helps depends on your model and hardware, so it is worth benchmarking your specific case.

We construct the solver components ourselves instead of using the simple `solve_ivp` interface, so that we can assemble the full solver and hand it to `torch.compile`.

Let's begin by importing everything we need in this example.

In [1]:
import torch
import torch.nn as nn
import torchode as to

torch.random.manual_seed(180819023);

Now we define a simple neural ODE given by an MLP with two hidden layers.

In [2]:
class Model(nn.Module):
    def __init__(self, n_features, n_hidden):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, n_hidden),
            nn.Softplus(),
            nn.Linear(n_hidden, n_hidden),
            nn.Softplus(),
            nn.Linear(n_hidden, n_features)
        )
    
    def forward(self, t, y):
        return self.layers(y)

n_features = 5
model = Model(n_features=n_features, n_hidden=32)

Next, we construct the solver components and then put them together into the solver `AutoDiffAdjoint` (that computes the parameter derivatives by backpropagating through the solver). We pass the model into the step method and step size controller so that it becomes part of the compiled solver.

In [3]:
dev = torch.device("cpu")
term = to.ODETerm(model)
step_method = to.Dopri5(term=term)
step_size_controller = to.IntegralController(atol=1e-6, rtol=1e-3, term=term)
adjoint = to.AutoDiffAdjoint(step_method, step_size_controller).to(dev)

Next, we compile the solver and our model.

In [4]:
adjoint_compiled = torch.compile(adjoint)

As a last step, we have to combine the initial condition `y0` and the evaluation points `t_eval` into a problem instance.

In [5]:
batch_size = 3
t_eval = torch.tile(torch.linspace(0.0, 3.0, 10), (batch_size, 1))
problem = to.InitialValueProblem(y0=torch.zeros((batch_size, 5)).to(dev), t_eval=t_eval.to(dev))

Here we see that both the eager and the compiled solver produce the same statistics and, up to small numerical differences from `torch.compile` reordering some operations, the same result.

In [6]:
sol = adjoint.solve(problem)
sol_compiled = adjoint_compiled.solve(problem)

print(sol.stats)
print(sol_compiled.stats)
diff = (sol.ys - sol_compiled.ys).detach().abs().max()
print("Max absolute difference", float(diff))

{'n_f_evals': tensor([38, 38, 38]), 'n_steps': tensor([6, 6, 6]), 'n_accepted': tensor([6, 6, 6]), 'n_initialized': tensor([10, 10, 10])}
{'n_f_evals': tensor([38, 38, 38]), 'n_steps': tensor([6, 6, 6]), 'n_accepted': tensor([6, 6, 6]), 'n_initialized': tensor([10, 10, 10])}
Max absolute difference 0.0
